# Which weather run reaches 08:00?

Reviewer point A1 asks for forecasts that were really available at 08:00 on D-1. Open-Meteo published the 00 UTC run at
06:51 UTC on 2026-10-31, after 08:00 Zurich time in summer. This notebook scores the 00 UTC run of D-1 and the 18 UTC run
of D-2 against ERA5 on 14 winter days and 14 summer days, all before the test year, to show what the safer run costs.

All data comes from the feature store through `WeatherRepository`, and the result goes to the result store.

In [1]:
import winterload.config  # Loads .env before anything imports fsspec.

import dataclasses
import datetime as dt

import pandas as pd
from winterload.forecast.evaluation.backtest import rows_of_local_days
from winterload.forecast.evaluation.weather_scores import score_forecast
from winterload.repository.factory import result_repository
from winterload.repository.results import WEATHER_RUN_CHOICE
from winterload.repository.weather import WeatherRepository
from winterload.storage import get_storage

ONE_DAY = dt.timedelta(days=1)
# The 00 UTC table has no January 2025, so the winter days sit in February.
SEASON_DAYS = {
    "winter": [dt.date(2025, 2, 1) + i * ONE_DAY for i in range(14)],
    "summer": [dt.date(2025, 7, 1) + i * ONE_DAY for i in range(14)],
}

weather = WeatherRepository(get_storage())
results = result_repository()

## Scores against ERA5

Columns: `run` is the weather run behind the forecast. `hours` counts the scored city-hours: 14 days x 24 hours x 4
cities = 1,344. The `*_mae` columns are mean absolute errors against ERA5: temperature in °C, cloud cover in
percentage points and shortwave radiation in W/m². `temperature_2m_mape_kelvin` is the temperature MAPE as a fraction,
computed in Kelvin because Celsius crosses zero in winter.

In [2]:
rows = []
for season, days in SEASON_DAYS.items():
    # A local day starts in the previous UTC day.
    start, end = days[0] - ONE_DAY, days[-1] + ONE_DAY
    truth = weather.reanalysis(start, end)
    for run, forecast in [("00 UTC D-1", weather.forecast_00utc(start, end)), ("18 UTC D-2", weather.forecast(start, end))]:
        score = score_forecast(rows_of_local_days(forecast, days), truth)
        rows.append({"season": season, "run": run, **dataclasses.asdict(score)})
result = pd.DataFrame(rows)
results.write(WEATHER_RUN_CHOICE, result)
result

,season,run,hours,temperature_2m_mae,cloud_cover_mae,shortwave_radiation_mae,temperature_2m_mape_kelvin
0,winter,00 UTC D-1,1344,1.480601,19.722470,13.262649,0.005346
1,winter,18 UTC D-2,1344,1.616801,19.218006,13.930804,0.005834
2,summer,00 UTC D-1,1344,1.335971,22.872024,39.647321,0.004529
3,summer,18 UTC D-2,1344,1.279518,22.970238,38.070685,0.004342
